In [ ]:
!pip install -q sentence-transformers faiss-cpu pyarrow transformers accelerate torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.8 MB/s eta 0:00:00


In [ ]:
import os
import re
import ast
import gc
import json
import math
import time
import warnings
import unicodedata
from pathlib import Path
from collections import Counter
from datetime import datetime

import numpy as np
import pandas as pd

import faiss
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 400)
pd.set_option("display.width", 240)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
LOCAL_FAISS_DIR = Path("artifacts/faiss")
DRIVE_FAISS_DIR = Path("/content/drive/MyDrive/tablewise/artifacts_new/faiss")

if DRIVE_FAISS_DIR.exists():
    FAISS_DIR = DRIVE_FAISS_DIR
elif LOCAL_FAISS_DIR.exists():
    FAISS_DIR = LOCAL_FAISS_DIR
else:
    raise FileNotFoundError(
        "Nu am găsit artifacts/faiss. Rulează mai întâi Notebookul 2."
    )

MAPPING_PATH = FAISS_DIR / "restaurant_index_mapping.parquet"
EMBEDDINGS_PATH = FAISS_DIR / "restaurant_embeddings.npy"
FAISS_INDEX_PATH = FAISS_DIR / "restaurant_faiss.index"
METADATA_PATH = FAISS_DIR / "metadata.json"

OUTPUT_DIR = FAISS_DIR.parent / "rag_generation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("FAISS_DIR:", FAISS_DIR)
print("MAPPING_PATH:", MAPPING_PATH)
print("EMBEDDINGS_PATH:", EMBEDDINGS_PATH)
print("FAISS_INDEX_PATH:", FAISS_INDEX_PATH)
print("METADATA_PATH:", METADATA_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

FAISS_DIR: /content/drive/MyDrive/tablewise/artifacts_new/faiss
MAPPING_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet
EMBEDDINGS_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy
FAISS_INDEX_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index
METADATA_PATH: /content/drive/MyDrive/tablewise/artifacts_new/faiss/metadata.json
OUTPUT_DIR: /content/drive/MyDrive/tablewise/artifacts_new/rag_generation


In [ ]:
assert MAPPING_PATH.exists(), f"Lipsește mapping-ul: {MAPPING_PATH}"
assert EMBEDDINGS_PATH.exists(), f"Lipsesc embeddings: {EMBEDDINGS_PATH}"
assert FAISS_INDEX_PATH.exists(), f"Lipsește FAISS index: {FAISS_INDEX_PATH}"

mapping_df = pd.read_parquet(MAPPING_PATH)
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

if METADATA_PATH.exists():
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        faiss_metadata = json.load(f)
else:
    faiss_metadata = {}

print("mapping_df shape:", mapping_df.shape)
print("embeddings shape:", embeddings.shape)
print("FAISS ntotal:", faiss_index.ntotal)

display(mapping_df.head(3))
faiss_metadata

mapping_df shape: (1033798, 31)
embeddings shape: (1033798, 384)
FAISS ntotal: 1033798


,faiss_idx,restaurant_id,name,country,region,province,city,city_filled,city_filled_norm,city_source,address,latitude,longitude,rating,price_level,price_bucket,top_tags_text,top_tags_norm_list,meals_text,meals_norm_list,features_text,features_norm_list,special_diets_text,special_diets_norm_list,popularity_detailed,popularity_rank_num,popularity_total_num,popularity_score,profile_quality_score,short_profile,profile_text
0,0,g10001637-d10002227,Le 147,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"10 Maison Neuve, 87510 Saint-Jouvent France",45.961674,1.169131,4.0,€,cheap,"Cheap Eats, French","[cheap eats, french]","Lunch, Dinner","[lunch, dinner]","Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Service","[reservations, seating, wheelchair accessible, serves alcohol, accepts credit cards, table service]",Unknown,[],#1 of 2 Restaurants in Saint-Jouvent,1.0,2.0,0.36907,11,"Le 147 | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats, French","Le 147 is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 10 Maison Neuve, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French. Meals served: Lunch, Dinner. Features: Reservations, Seating, Wheelchair Accessible, Serves Alcohol, Accepts Credit Cards, Table Servi..."
1,1,g10001637-d14975787,Le Saint Jouvent,France,Nouvelle-Aquitaine,Haute-Vienne,Saint-Jouvent,Saint-Jouvent,saint-jouvent,original_city,"16 Place de l Eglise, 87510 Saint-Jouvent France",45.957040,1.205480,4.0,€,cheap,Cheap Eats,[cheap eats],Unknown,[],Unknown,[],Unknown,[],#2 of 2 Restaurants in Saint-Jouvent,2.0,2.0,0.00000,9,"Le Saint Jouvent | Saint-Jouvent, France | rating=4.0 | price=cheap | tags=Cheap Eats","Le Saint Jouvent is a restaurant located in Saint-Jouvent, Haute-Vienne, Nouvelle-Aquitaine, France. Address: 16 Place de l Eglise, 87510 Saint-Jouvent France. Average rating: 4.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats. Popularity: #2 of 2 Restaurants in Saint-Jouvent."
2,2,g10002858-d4586832,Au Bout du Pont,France,Centre-Val de Loire,Berry,Rivarennes,Rivarennes,rivarennes,original_city,"2 rue des Dames, 36800 Rivarennes France",46.635895,1.386133,5.0,€,cheap,"Cheap Eats, French, European","[cheap eats, french, european]","Dinner, Lunch, Drinks","[dinner, lunch, drinks]","Reservations, Seating, Table Service, Wheelchair Accessible","[reservations, seating, table service, wheelchair accessible]",Unknown,[],#1 of 1 Restaurant in Rivarennes,1.0,1.0,NaN,11,"Au Bout du Pont | Rivarennes, France | rating=5.0 | price=cheap | tags=Cheap Eats, French, European","Au Bout du Pont is a restaurant located in Rivarennes, Berry, Centre-Val de Loire, France. Address: 2 rue des Dames, 36800 Rivarennes France. Average rating: 5.0 out of 5. Price category: cheap. Original price level: €. Cuisines and tags: Cheap Eats, French, European. Meals served: Dinner, Lunch, Drinks. Features: Reservations, Seating, Table Service, Wheelchair Accessible. Popularity: #1 of 1..."


{'created_at': '2026-05-04T16:00:35.667690Z',
 'input_path': '/content/drive/MyDrive/tablewise/data_new/processed/restaurants_indexable.parquet',
 'embedding_data_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurants_for_embeddings.parquet',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'num_restaurants': 1033798,
 'embedding_dim': 384,
 'normalized_embeddings': True,
 'faiss_index_type': 'IndexFlatIP',
 'use_sample': False,
 'sample_size': None,
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'faiss_index_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_faiss.index',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'python_version': '3.12.13',
 'numpy_version': '2.0.2',
 'pandas_version': '2.2.2'}

In [ ]:
required_cols = [
    "faiss_idx",
    "restaurant_id",
    "name",
    "country",
    "city_filled",
    "city_filled_norm",
    "price_bucket",
    "rating",
    "popularity_score",
    "profile_text",
    "short_profile",
]

missing_cols = [c for c in required_cols if c not in mapping_df.columns]
assert not missing_cols, f"Lipsesc coloane obligatorii din mapping: {missing_cols}"

assert len(mapping_df) == embeddings.shape[0], "mapping_df și embeddings au dimensiuni diferite."
assert len(mapping_df) == faiss_index.ntotal, "mapping_df și FAISS index au dimensiuni diferite."
assert mapping_df["faiss_idx"].is_unique, "faiss_idx nu este unic."
assert mapping_df["faiss_idx"].min() == 0, "faiss_idx nu începe de la 0."
assert mapping_df["faiss_idx"].max() == len(mapping_df) - 1, "faiss_idx nu este continuu."

print("Validări OK.")

Validări OK.


In [ ]:
EMBEDDING_MODEL_NAME = faiss_metadata.get(
    "embedding_model",
    "sentence-transformers/all-MiniLM-L6-v2",
)

print("Loading embedding model:", EMBEDDING_MODEL_NAME)

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Model loaded.")

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded.


In [ ]:
def strip_accents(text):
    if text is None or pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))

    return text


def normalize_text(x):
    if x is None or pd.isna(x):
        return ""

    s = strip_accents(str(x)).lower().strip()
    s = re.sub(r"[^a-z0-9\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return s


def safe_value(x, default="Unknown"):
    if x is None or pd.isna(x):
        return default

    s = str(x).strip()

    if not s or s.lower() in {"nan", "none", "null"}:
        return default

    return s


def contains_phrase(text_norm, phrase_norm):
    if not text_norm or not phrase_norm:
        return False

    pattern = rf"(?<![a-z0-9]){re.escape(phrase_norm)}(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None

In [ ]:
def parse_possible_list(x):
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, np.ndarray):
        values = x.tolist()
    elif isinstance(x, (list, tuple, set)):
        values = list(x)
    else:
        s = str(x).strip()

        if not s or s.lower() in {"nan", "none", "null", "unknown"}:
            return []

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    values = list(parsed)
                else:
                    values = [s]
            except Exception:
                values = re.split(r"[,|;/]", s)
        else:
            values = re.split(r"[,|;/]", s)

    cleaned = []
    seen = set()

    for item in values:
        item_s = safe_value(item, default="").strip()
        item_norm = normalize_text(item_s)

        if item_norm and item_norm not in seen:
            cleaned.append(item_s)
            seen.add(item_norm)

    return cleaned

In [ ]:
df = mapping_df.copy()

list_base_cols = [
    "top_tags",
    "meals",
    "features",
    "special_diets",
]

for base in list_base_cols:
    text_col = f"{base}_text"
    list_col = f"{base}_list"
    norm_list_col = f"{base}_norm_list"

    if list_col not in df.columns:
        if text_col in df.columns:
            df[list_col] = df[text_col].apply(parse_possible_list)
        else:
            df[list_col] = [[] for _ in range(len(df))]
    else:
        df[list_col] = df[list_col].apply(parse_possible_list)

    if norm_list_col not in df.columns:
        df[norm_list_col] = df[list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )
    else:
        df[norm_list_col] = df[norm_list_col].apply(parse_possible_list)
        df[norm_list_col] = df[norm_list_col].apply(
            lambda xs: [normalize_text(x) for x in xs if normalize_text(x)]
        )

for col in ["country", "region", "province", "city", "city_filled"]:
    if col in df.columns:
        df[f"{col}_norm"] = df[col].apply(normalize_text)

df["city_filled_norm"] = df["city_filled"].apply(normalize_text)
df["country_norm"] = df["country"].apply(normalize_text)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")
df["popularity_score"] = pd.to_numeric(df["popularity_score"], errors="coerce")

print("Prepared df shape:", df.shape)
display(df[["name", "country", "city_filled", "rating", "price_bucket", "popularity_score", "top_tags_norm_list"]].head(3))

Prepared df shape: (1033798, 39)


,name,country,city_filled,rating,price_bucket,popularity_score,top_tags_norm_list
0,Le 147,France,Saint-Jouvent,4.0,cheap,0.36907,"[cheap eats, french]"
1,Le Saint Jouvent,France,Saint-Jouvent,4.0,cheap,0.00000,[cheap eats]
2,Au Bout du Pont,France,Rivarennes,5.0,cheap,NaN,"[cheap eats, french, european]"


In [ ]:
BAD_CITY_TERMS = {
    "",
    "nan",
    "none",
    "null",
    "unknown",
    "street",
    "st",
    "road",
    "rd",
    "avenue",
    "ave",
    "square",
    "center",
    "centre",
    "park",
    "hotel",
    "restaurant",
    "restaurants",
    "best",
    "cheap",
    "budget",
    "expensive",
    "food",
    "pub",
    "bar",
    "cafe",
    "cafes",
    "coffee",
    "pizza",
    "pasta",
    "sushi",
    "seafood",
    "breakfast",
    "brunch",
    "lunch",
    "dinner",
    "europe",
    "united kingdom uk",
    "united kingdom",
    "uk",
    "england",
    "scotland",
    "wales",
    "northern ireland",
    "france",
    "italy",
    "spain",
    "germany",
    "portugal",
    "greece",
}

country_vocab = sorted(
    {
        normalize_text(x)
        for x in df["country_norm"].dropna().tolist()
        if normalize_text(x) not in {"", "unknown", "nan", "none", "null"}
    },
    key=len,
    reverse=True,
)

country_values = set(df["country_norm"].dropna().astype(str).map(normalize_text))
region_values = set(df["region_norm"].dropna().astype(str).map(normalize_text)) if "region_norm" in df.columns else set()
province_values = set(df["province_norm"].dropna().astype(str).map(normalize_text)) if "province_norm" in df.columns else set()

blocked_geo_values = country_values | region_values | province_values | BAD_CITY_TERMS
city_counts = df["city_filled_norm"].value_counts(dropna=True).to_dict()

raw_city_values = (
    df["city_filled_norm"]
    .dropna()
    .astype(str)
    .map(normalize_text)
    .tolist()
)

city_vocab = []

for city in sorted(set(raw_city_values), key=len, reverse=True):
    if not city:
        continue
    if city in blocked_geo_values:
        continue
    if len(city) < 2:
        continue
    if city.isdigit():
        continue
    if len(city.split()) > 6:
        continue

    city_vocab.append(city)

IMPORTANT_CITY_ALIASES_RAW = {
    "rome": ["rome", "roma"],
    "paris": ["paris"],
    "madrid": ["madrid"],
    "barcelona": ["barcelona"],
    "milan": ["milan", "milano"],
    "berlin": ["berlin"],
    "lisbon": ["lisbon", "lisboa"],
    "athens": ["athens", "athina", "athena"],
    "london": ["london", "londra"],
    "manchester": ["manchester"],
    "edinburgh": ["edinburgh"],
    "dublin": ["dublin"],
    "vienna": ["vienna", "wien"],
    "prague": ["prague", "praha"],
    "amsterdam": ["amsterdam"],
    "brussels": ["brussels", "bruxelles", "brussel"],
    "budapest": ["budapest"],
    "warsaw": ["warsaw", "warszawa"],
    "krakow": ["krakow", "kraków"],
    "porto": ["porto", "oporto"],
    "florence": ["florence", "firenze"],
    "venice": ["venice", "venezia"],
    "naples": ["naples", "napoli"],
    "turin": ["turin", "torino"],
    "seville": ["seville", "sevilla"],
    "valencia": ["valencia"],
    "granada": ["granada"],
    "malaga": ["malaga", "málaga"],
    "nice": ["nice"],
    "lyon": ["lyon"],
    "marseille": ["marseille"],
    "bordeaux": ["bordeaux"],
    "hamburg": ["hamburg"],
    "munich": ["munich", "muenchen", "münchen"],
    "cologne": ["cologne", "koln", "köln"],
    "frankfurt": ["frankfurt"],
}

CITY_ALIASES = {}

for canonical, aliases in IMPORTANT_CITY_ALIASES_RAW.items():
    canonical_norm = normalize_text(canonical)
    alias_norms = [normalize_text(a) for a in aliases]

    if canonical_norm not in city_vocab:
        city_vocab.append(canonical_norm)

    for alias_norm in alias_norms:
        CITY_ALIASES[alias_norm] = canonical_norm

important_order = list(IMPORTANT_CITY_ALIASES_RAW.keys())

city_vocab = sorted(
    set(city_vocab),
    key=lambda c: (
        0 if c in important_order else 1,
        important_order.index(c) if c in important_order else 999999,
        -len(c),
    ),
)

print("Countries:", len(country_vocab))
print("Cities:", len(city_vocab))

print("\nCity checks:")
for c in ["rome", "paris", "madrid", "barcelona", "milan", "berlin", "lisbon", "athens", "london"]:
    print(c, "in vocab:", c in city_vocab, "| rows:", int((df["city_filled_norm"] == c).sum()))

Countries: 24
Cities: 62251

City checks:
rome in vocab: True | rows: 12603
paris in vocab: True | rows: 18126
madrid in vocab: True | rows: 12133
barcelona in vocab: True | rows: 10283
milan in vocab: True | rows: 8381
berlin in vocab: True | rows: 0
lisbon in vocab: True | rows: 5261
athens in vocab: True | rows: 2915
london in vocab: True | rows: 0


In [ ]:
def flatten_norm_list_column(series):
    values = Counter()

    for xs in series:
        if not isinstance(xs, list):
            continue

        for x in xs:
            x_norm = normalize_text(x)

            if x_norm:
                values[x_norm] += 1

    return values


tag_counter = flatten_norm_list_column(df["top_tags_norm_list"]) if "top_tags_norm_list" in df.columns else Counter()
meal_counter = flatten_norm_list_column(df["meals_norm_list"]) if "meals_norm_list" in df.columns else Counter()
feature_counter = flatten_norm_list_column(df["features_norm_list"]) if "features_norm_list" in df.columns else Counter()
diet_counter = flatten_norm_list_column(df["special_diets_norm_list"]) if "special_diets_norm_list" in df.columns else Counter()

tag_vocab = sorted(tag_counter.keys(), key=len, reverse=True)
meal_vocab = sorted(meal_counter.keys(), key=len, reverse=True)
feature_vocab = sorted(feature_counter.keys(), key=len, reverse=True)
diet_vocab = sorted(diet_counter.keys(), key=len, reverse=True)

print("tag_vocab:", len(tag_vocab))
print("meal_vocab:", len(meal_vocab))
print("feature_vocab:", len(feature_vocab))
print("diet_vocab:", len(diet_vocab))

tag_vocab: 189
meal_vocab: 6
feature_vocab: 39
diet_vocab: 5


In [ ]:
COUNTRY_ALIASES = {
    "uk": "united kingdom",
    "u k": "united kingdom",
    "great britain": "united kingdom",
    "britain": "united kingdom",
    "england": "united kingdom",
    "scotland": "united kingdom",
    "wales": "united kingdom",
    "northern ireland": "united kingdom",
    "deutschland": "germany",
    "espana": "spain",
    "españa": "spain",
    "italia": "italy",
    "grecia": "greece",
}

PRICE_KEYWORDS = {
    "cheap": [
        "cheap",
        "budget",
        "affordable",
        "low cost",
        "low-cost",
        "inexpensive",
        "not expensive",
        "economic",
        "economical",
        "ieftin",
        "ieftina",
        "ieftine",
    ],
    "mid": [
        "mid range",
        "mid-range",
        "moderate",
        "casual",
        "normal price",
        "average price",
        "medium price",
        "mediu",
        "pret mediu",
        "preț mediu",
    ],
    "expensive": [
        "expensive",
        "fine dining",
        "luxury",
        "high end",
        "high-end",
        "premium",
        "fancy",
        "upscale",
        "scump",
        "scumpa",
        "scumpe",
        "lux",
    ],
}

TAG_SYNONYMS = {
    "italian": ["italian", "italian food", "pizza", "pasta", "trattoria", "osteria"],
    "french": ["french", "french food", "bistro", "brasserie"],
    "spanish": ["spanish", "spanish food", "tapas", "paella"],
    "greek": ["greek", "greek food", "taverna", "souvlaki", "gyros"],
    "portuguese": ["portuguese", "portuguese food"],
    "german": ["german", "german food"],
    "seafood": ["seafood", "fish", "oyster", "sushi"],
    "steakhouse": ["steakhouse", "steak", "grill"],
    "asian": ["asian", "chinese", "japanese", "thai", "vietnamese", "korean"],
    "indian": ["indian", "curry"],
    "mexican": ["mexican", "tacos", "burrito"],
    "mediterranean": ["mediterranean"],
    "fast food": ["fast food", "burger", "burgers", "kebab"],
    "coffee": ["coffee", "cafe", "cafes", "café"],
    "bar": ["bar", "pub", "drinks", "cocktails"],
}

DIETARY_SYNONYMS = {
    "vegetarian friendly": ["vegetarian", "veggie", "vegetarian friendly", "fara carne", "fără carne"],
    "vegan options": ["vegan", "vegan options", "plant based", "plant-based"],
    "gluten free options": ["gluten free", "gluten-free", "without gluten", "fara gluten", "fără gluten"],
}

MEAL_SYNONYMS = {
    "breakfast": ["breakfast", "mic dejun"],
    "brunch": ["brunch"],
    "lunch": ["lunch", "pranz", "prânz"],
    "dinner": ["dinner", "cina", "cină"],
    "drinks": ["drinks", "cocktails", "beer", "wine"],
}

FEATURE_SYNONYMS = {
    "reservations": ["reservation", "reservations", "booking", "book a table", "rezervare"],
    "outdoor seating": ["outdoor", "terrace", "terrace seating", "outdoor seating", "terasă", "terasa"],
    "delivery": ["delivery", "takeaway", "take out", "takeout"],
    "wheelchair accessible": ["wheelchair", "accessible", "wheelchair accessible"],
    "family friendly": ["family friendly", "kid friendly", "kids", "children", "familie", "copii"],
    "romantic": ["romantic", "date night", "cozy", "cosy"],
}

In [ ]:
def find_synonym_matches(query_norm, synonym_dict):
    found = []

    for canonical, synonyms in synonym_dict.items():
        for syn in synonyms:
            syn_norm = normalize_text(syn)

            if contains_phrase(query_norm, syn_norm):
                found.append(canonical)
                break

    return sorted(set(found))


def extract_price_bucket(query_norm):
    for bucket, keywords in PRICE_KEYWORDS.items():
        for kw in keywords:
            kw_norm = normalize_text(kw)

            if contains_phrase(query_norm, kw_norm):
                return bucket

    return None


def extract_vocab_matches(query_norm, vocab, max_matches=8, min_len=3):
    found = []

    for term in vocab:
        term_norm = normalize_text(term)

        if len(term_norm) < min_len:
            continue

        if contains_phrase(query_norm, term_norm):
            found.append(term_norm)

        if len(found) >= max_matches:
            break

    return sorted(set(found))


def extract_country(query_norm):
    for alias, canonical in COUNTRY_ALIASES.items():
        alias_norm = normalize_text(alias)

        if contains_phrase(query_norm, alias_norm):
            canonical_norm = normalize_text(canonical)

            if canonical_norm in country_vocab:
                return canonical_norm

    for country in country_vocab:
        if contains_phrase(query_norm, country):
            return country

    return None


def city_has_location_context(query_norm, city_norm):
    if not contains_phrase(query_norm, city_norm):
        return False

    prep_patterns = [
        rf"\bin\s+{re.escape(city_norm)}\b",
        rf"\bnear\s+{re.escape(city_norm)}\b",
        rf"\baround\s+{re.escape(city_norm)}\b",
        rf"\bat\s+{re.escape(city_norm)}\b",
        rf"\bfrom\s+{re.escape(city_norm)}\b",
        rf"\bin apropiere de\s+{re.escape(city_norm)}\b",
        rf"\blanga\s+{re.escape(city_norm)}\b",
        rf"\blângă\s+{re.escape(city_norm)}\b",
        rf"\bin\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orasul\s+{re.escape(city_norm)}\b",
        rf"\bîn\s+orașul\s+{re.escape(city_norm)}\b",
    ]

    for pattern in prep_patterns:
        if re.search(pattern, query_norm):
            return True

    if len(city_norm) >= 4 and city_norm not in BAD_CITY_TERMS:
        return True

    return False


def extract_city(query_norm):
    for alias, canonical in CITY_ALIASES.items():
        alias_norm = normalize_text(alias)
        canonical_norm = normalize_text(canonical)

        if contains_phrase(query_norm, alias_norm) and canonical_norm in city_vocab:
            return canonical_norm

    candidates = []

    for city in city_vocab:
        if city_has_location_context(query_norm, city):
            candidates.append(city)

    if not candidates:
        return None

    return max(candidates, key=len)


def parse_user_query(query):
    query_norm = normalize_text(query)

    city = extract_city(query_norm)
    country = extract_country(query_norm)
    price_bucket = extract_price_bucket(query_norm)

    tags_from_synonyms = find_synonym_matches(query_norm, TAG_SYNONYMS)
    dietary_from_synonyms = find_synonym_matches(query_norm, DIETARY_SYNONYMS)
    meals_from_synonyms = find_synonym_matches(query_norm, MEAL_SYNONYMS)
    features_from_synonyms = find_synonym_matches(query_norm, FEATURE_SYNONYMS)

    tags_from_vocab = extract_vocab_matches(query_norm, tag_vocab, max_matches=8)
    meals_from_vocab = extract_vocab_matches(query_norm, meal_vocab, max_matches=5)
    features_from_vocab = extract_vocab_matches(query_norm, feature_vocab, max_matches=5)
    diets_from_vocab = extract_vocab_matches(query_norm, diet_vocab, max_matches=5)

    parsed = {
        "original_query": query,
        "normalized_query": query_norm,
        "city": city,
        "country": country,
        "price_bucket": price_bucket,
        "tags": sorted(set(tags_from_synonyms + tags_from_vocab)),
        "dietary": sorted(set(dietary_from_synonyms + diets_from_vocab)),
        "matched_meals": sorted(set(meals_from_synonyms + meals_from_vocab)),
        "matched_features": sorted(set(features_from_synonyms + features_from_vocab)),
    }

    return parsed

In [ ]:
def list_contains_term(row_list, term):
    if not isinstance(row_list, list):
        return False

    term_norm = normalize_text(term)

    if not term_norm:
        return False

    for item in row_list:
        item_norm = normalize_text(item)

        if not item_norm:
            continue

        if term_norm == item_norm:
            return True

        if term_norm in item_norm:
            return True

        if item_norm in term_norm and len(item_norm) >= 4:
            return True

    return False


def list_contains_any(row_list, terms):
    return any(list_contains_term(row_list, term) for term in terms)

In [ ]:
def apply_location_filter(df_in, parsed_query, verbose=True):
    current = df_in.copy()

    city = parsed_query.get("city")
    country = parsed_query.get("country")

    if city:
        before = len(current)
        filtered = current[current["city_filled_norm"] == city].copy()

        if verbose:
            print(f"City filter city={city}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print("Oraș detectat, dar fără rezultate în dataset. Returnăm subset gol.")
            return filtered

        current = filtered

    if country:
        before = len(current)
        filtered = current[current["country_norm"] == country].copy()

        if verbose:
            print(f"Country filter country={country}: {before} -> {len(filtered)}")

        if len(filtered) == 0:
            if verbose:
                print("Țară detectată, dar fără rezultate în dataset. Returnăm subset gol.")
            return filtered

        current = filtered

    return current


def compute_metadata_score_for_row(row, parsed_query):
    score = 0.0
    reasons = []

    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if price_bucket and row.get("price_bucket") == price_bucket:
        score += 2.0
        reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        if list_contains_term(row_tags, tag):
            score += 2.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            score += 2.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            score += 1.0
            reasons.append(f"feature={feature}")

    return score, sorted(set(reasons))


def add_metadata_scores(df_in, parsed_query):
    current = df_in.copy()

    if len(current) == 0:
        current["metadata_score"] = pd.Series(dtype=float)
        current["metadata_match_reasons"] = pd.Series(dtype=object)
        return current

    scores_and_reasons = current.apply(
        lambda row: compute_metadata_score_for_row(row, parsed_query),
        axis=1,
    )

    current["metadata_score"] = scores_and_reasons.apply(lambda x: x[0])
    current["metadata_match_reasons"] = scores_and_reasons.apply(lambda x: x[1])

    return current


def apply_metadata_filter(df_in, parsed_query, min_results=50, verbose=True):
    current = add_metadata_scores(df_in, parsed_query)

    if len(current) == 0:
        return current

    has_constraints = bool(
        parsed_query.get("price_bucket")
        or parsed_query.get("tags")
        or parsed_query.get("dietary")
        or parsed_query.get("matched_meals")
        or parsed_query.get("matched_features")
    )

    if not has_constraints:
        if verbose:
            print("No metadata constraints.")
        return current

    before = len(current)
    filtered = current[current["metadata_score"] > 0].copy()

    if verbose:
        print(f"Metadata score > 0 filter: {before} -> {len(filtered)}")

    if len(filtered) >= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if 0 < len(filtered) < min_results and before <= min_results:
        return filtered.sort_values("metadata_score", ascending=False).reset_index(drop=True)

    if verbose:
        print("Metadata filter prea strict. Păstrăm subsetul curent, dar folosim metadata_score la ranking.")

    return current.sort_values("metadata_score", ascending=False).reset_index(drop=True)


def filter_candidates(query, df_in, min_metadata_results=50, verbose=True):
    parsed = parse_user_query(query)

    if verbose:
        print("Parsed query:")
        print(json.dumps(parsed, indent=2, ensure_ascii=False))
        print("\nInitial rows:", len(df_in))

    candidates = apply_location_filter(
        df_in,
        parsed,
        verbose=verbose,
    )

    if verbose:
        print("After location filter:", len(candidates))

    if len(candidates) == 0:
        candidates = add_metadata_scores(candidates, parsed)
        return parsed, candidates.reset_index(drop=True)

    candidates = apply_metadata_filter(
        candidates,
        parsed,
        min_results=min_metadata_results,
        verbose=verbose,
    )

    if verbose:
        print("After metadata filter/scoring:", len(candidates))

    return parsed, candidates.reset_index(drop=True)

In [ ]:
def encode_query(query):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    return query_embedding[0]


def semantic_search_within_candidates(
    query,
    candidates_df,
    embeddings_array,
    top_k=100,
    score_chunk_size=200_000,
    verbose=True,
):
    if len(candidates_df) == 0:
        result = candidates_df.copy()
        result["semantic_score"] = pd.Series(dtype=float)
        return result

    query_embedding = encode_query(query)

    faiss_indices = candidates_df["faiss_idx"].to_numpy(dtype=np.int64)

    assert faiss_indices.min() >= 0
    assert faiss_indices.max() < embeddings_array.shape[0]

    n = len(faiss_indices)
    scores = np.empty(n, dtype=np.float32)

    if verbose:
        print(f"Computing semantic scores for {n:,} candidates...")

    for start in range(0, n, score_chunk_size):
        end = min(start + score_chunk_size, n)

        idx_chunk = faiss_indices[start:end]
        emb_chunk = embeddings_array[idx_chunk]

        scores[start:end] = emb_chunk @ query_embedding

    k = min(top_k, n)

    if k == n:
        top_positions = np.argsort(-scores)
    else:
        top_positions_unsorted = np.argpartition(-scores, k - 1)[:k]
        top_positions = top_positions_unsorted[np.argsort(-scores[top_positions_unsorted])]

    result = candidates_df.iloc[top_positions].copy().reset_index(drop=True)
    result["semantic_score"] = scores[top_positions]

    return result


def retrieve_candidates_for_reranking(
    query,
    df_in,
    embeddings_array,
    candidate_min_metadata_results=50,
    semantic_top_k=100,
    score_chunk_size=200_000,
    verbose=True,
):
    start_time = time.time()

    parsed, candidates = filter_candidates(
        query=query,
        df_in=df_in,
        min_metadata_results=candidate_min_metadata_results,
        verbose=verbose,
    )

    if len(candidates) == 0:
        empty_results = candidates.copy()
        empty_results["semantic_score"] = pd.Series(dtype=float)

        return {
            "query": query,
            "parsed_query": parsed,
            "num_candidates_after_filter": 0,
            "candidates": empty_results,
            "elapsed_seconds": time.time() - start_time,
        }

    semantic_results = semantic_search_within_candidates(
        query=query,
        candidates_df=candidates,
        embeddings_array=embeddings_array,
        top_k=semantic_top_k,
        score_chunk_size=score_chunk_size,
        verbose=verbose,
    )

    elapsed = time.time() - start_time

    return {
        "query": query,
        "parsed_query": parsed,
        "num_candidates_after_filter": len(candidates),
        "candidates": semantic_results,
        "elapsed_seconds": elapsed,
    }

In [ ]:
def normalize_semantic_score(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)
    return ((s + 1.0) / 2.0).clip(0, 1)


def normalize_metadata_score(series):
    s = pd.to_numeric(series, errors="coerce").fillna(0.0)

    max_val = s.max()

    if pd.isna(max_val) or max_val <= 0:
        return pd.Series(np.zeros(len(s)), index=s.index)

    return (s / max_val).clip(0, 1)


def normalize_rating_score(series):
    s = pd.to_numeric(series, errors="coerce")
    normalized = (s / 5.0).clip(0, 1)
    normalized = normalized.fillna(0.5)

    return normalized


def normalize_popularity_score(series):
    s = pd.to_numeric(series, errors="coerce")
    return s.clip(0, 1).fillna(0.5)


def compute_constraint_scores_for_row(row, parsed_query):
    hard_score = 1.0
    soft_score = 0.0
    max_soft_score = 0.0
    reasons = []

    city = parsed_query.get("city")
    country = parsed_query.get("country")
    price_bucket = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    if city:
        if row.get("city_filled_norm") == city:
            reasons.append(f"city={city}")
        else:
            hard_score = 0.0

    if country:
        if row.get("country_norm") == country:
            reasons.append(f"country={country}")
        else:
            hard_score = 0.0

    if price_bucket:
        max_soft_score += 1.0
        if row.get("price_bucket") == price_bucket:
            soft_score += 1.0
            reasons.append(f"price={price_bucket}")

    row_tags = row.get("top_tags_norm_list", [])
    row_diets = row.get("special_diets_norm_list", [])
    row_meals = row.get("meals_norm_list", [])
    row_features = row.get("features_norm_list", [])

    for tag in tags:
        max_soft_score += 1.0
        if list_contains_term(row_tags, tag):
            soft_score += 1.0
            reasons.append(f"tag={tag}")

    for diet in dietary:
        max_soft_score += 1.0
        if (
            list_contains_term(row_diets, diet)
            or list_contains_term(row_tags, diet)
            or list_contains_term(row_features, diet)
        ):
            soft_score += 1.0
            reasons.append(f"diet={diet}")

    for meal in meals:
        max_soft_score += 1.0
        if list_contains_term(row_meals, meal) or list_contains_term(row_tags, meal):
            soft_score += 1.0
            reasons.append(f"meal={meal}")

    for feature in features:
        max_soft_score += 1.0
        if list_contains_term(row_features, feature) or list_contains_term(row_tags, feature):
            soft_score += 1.0
            reasons.append(f"feature={feature}")

    if max_soft_score > 0:
        soft_constraint_score = soft_score / max_soft_score
    else:
        soft_constraint_score = 0.5

    return hard_score, soft_constraint_score, sorted(set(reasons))

In [ ]:
RERANK_WEIGHTS = {
    "semantic": 0.40,
    "metadata": 0.20,
    "rating": 0.15,
    "popularity": 0.10,
    "soft_constraints": 0.15,
}

assert abs(sum(RERANK_WEIGHTS.values()) - 1.0) < 1e-9


def rerank_candidates(candidates_df, parsed_query, top_k=20):
    results = candidates_df.copy()

    if len(results) == 0:
        results["semantic_score_norm"] = pd.Series(dtype=float)
        results["metadata_score_norm"] = pd.Series(dtype=float)
        results["rating_score"] = pd.Series(dtype=float)
        results["popularity_score_norm"] = pd.Series(dtype=float)
        results["hard_constraint_score"] = pd.Series(dtype=float)
        results["soft_constraint_score"] = pd.Series(dtype=float)
        results["constraint_match_reasons"] = pd.Series(dtype=object)
        results["final_rerank_score"] = pd.Series(dtype=float)
        return results

    if "metadata_score" not in results.columns:
        results["metadata_score"] = 0.0

    results["semantic_score_norm"] = normalize_semantic_score(results["semantic_score"])
    results["metadata_score_norm"] = normalize_metadata_score(results["metadata_score"])
    results["rating_score"] = normalize_rating_score(results["rating"])
    results["popularity_score_norm"] = normalize_popularity_score(results["popularity_score"])

    constraint_values = results.apply(
        lambda row: compute_constraint_scores_for_row(row, parsed_query),
        axis=1,
    )

    results["hard_constraint_score"] = constraint_values.apply(lambda x: x[0])
    results["soft_constraint_score"] = constraint_values.apply(lambda x: x[1])
    results["constraint_match_reasons"] = constraint_values.apply(lambda x: x[2])

    results["final_rerank_score"] = (
        RERANK_WEIGHTS["semantic"] * results["semantic_score_norm"]
        + RERANK_WEIGHTS["metadata"] * results["metadata_score_norm"]
        + RERANK_WEIGHTS["rating"] * results["rating_score"]
        + RERANK_WEIGHTS["popularity"] * results["popularity_score_norm"]
        + RERANK_WEIGHTS["soft_constraints"] * results["soft_constraint_score"]
    )

    results["final_rerank_score"] = (
        results["final_rerank_score"] * results["hard_constraint_score"]
    )

    results = results.sort_values(
        [
            "final_rerank_score",
            "soft_constraint_score",
            "metadata_score_norm",
            "semantic_score_norm",
            "rating_score",
            "popularity_score_norm",
        ],
        ascending=False,
    ).reset_index(drop=True)

    if top_k is not None:
        results = results.head(top_k).copy().reset_index(drop=True)

    results.insert(0, "rerank_position", np.arange(1, len(results) + 1))

    return results


def retrieve_and_rerank(
    query,
    df_in,
    embeddings_array,
    retrieval_top_k=100,
    rerank_top_k=20,
    candidate_min_metadata_results=50,
    verbose=True,
):
    start_time = time.time()

    retrieval_output = retrieve_candidates_for_reranking(
        query=query,
        df_in=df_in,
        embeddings_array=embeddings_array,
        candidate_min_metadata_results=candidate_min_metadata_results,
        semantic_top_k=retrieval_top_k,
        verbose=verbose,
    )

    parsed_query = retrieval_output["parsed_query"]
    candidates = retrieval_output["candidates"]

    reranked = rerank_candidates(
        candidates_df=candidates,
        parsed_query=parsed_query,
        top_k=rerank_top_k,
    )

    elapsed = time.time() - start_time

    return {
        "query": query,
        "parsed_query": parsed_query,
        "num_candidates_after_filter": retrieval_output["num_candidates_after_filter"],
        "num_retrieved_for_reranking": len(candidates),
        "results": reranked,
        "elapsed_seconds": elapsed,
    }

In [ ]:
def format_rating(x):
    if pd.isna(x):
        return "unknown"
    try:
        return f"{float(x):.1f}/5"
    except Exception:
        return "unknown"


def format_price_bucket(x):
    x = safe_value(x, default="unknown").lower()
    if x in {"cheap", "mid", "expensive"}:
        return x
    return "unknown"


def compact_text(x, max_chars=300):
    s = safe_value(x, default="")
    s = re.sub(r"\s+", " ", s).strip()

    if len(s) <= max_chars:
        return s

    return s[:max_chars].rstrip() + "..."


def row_to_rag_evidence(row, rank=None):
    name = safe_value(row.get("name"))
    city = safe_value(row.get("city_filled"))
    country = safe_value(row.get("country"))
    rating = format_rating(row.get("rating"))
    price = format_price_bucket(row.get("price_bucket"))

    tags = safe_value(row.get("top_tags_text"))
    diets = safe_value(row.get("special_diets_text"))
    meals = safe_value(row.get("meals_text"))
    features = safe_value(row.get("features_text"))
    popularity = safe_value(row.get("popularity_detailed"))
    reasons = row.get("constraint_match_reasons", [])
    final_score = row.get("final_rerank_score", np.nan)

    if not isinstance(reasons, list):
        reasons = []

    evidence = {
        "rank": int(rank) if rank is not None else None,
        "name": name,
        "location": f"{city}, {country}",
        "city": city,
        "country": country,
        "rating": rating,
        "price_bucket": price,
        "cuisines_and_tags": tags,
        "special_diets": diets,
        "meals": meals,
        "features": features,
        "popularity": popularity,
        "match_reasons": reasons,
        "final_rerank_score": None if pd.isna(final_score) else float(final_score),
    }

    return evidence

In [ ]:
def build_rag_context(reranked_df, top_n=5):
    if len(reranked_df) == 0:
        return {
            "restaurants": [],
            "context_text": "",
            "allowed_restaurant_names": [],
        }

    top_df = reranked_df.head(top_n).copy()

    restaurants = []

    for i, (_, row) in enumerate(top_df.iterrows(), start=1):
        restaurants.append(row_to_rag_evidence(row, rank=i))

    context_lines = []

    for r in restaurants:
        context_lines.append(f"[{r['rank']}] {r['name']}")
        context_lines.append(f"Location: {r['location']}")
        context_lines.append(f"Rating: {r['rating']}")
        context_lines.append(f"Price bucket: {r['price_bucket']}")

        if r["cuisines_and_tags"] != "Unknown":
            context_lines.append(f"Cuisines/tags: {r['cuisines_and_tags']}")

        if r["special_diets"] != "Unknown":
            context_lines.append(f"Special diets: {r['special_diets']}")

        if r["meals"] != "Unknown":
            context_lines.append(f"Meals: {r['meals']}")

        if r["features"] != "Unknown":
            context_lines.append(f"Features: {r['features']}")

        if r["popularity"] != "Unknown":
            context_lines.append(f"Popularity: {r['popularity']}")

        if r["match_reasons"]:
            context_lines.append(f"Matched constraints: {', '.join(r['match_reasons'])}")

        context_lines.append("")

    context_text = "\n".join(context_lines).strip()

    allowed_restaurant_names = [r["name"] for r in restaurants]

    return {
        "restaurants": restaurants,
        "context_text": context_text,
        "allowed_restaurant_names": allowed_restaurant_names,
    }

In [ ]:
test_query = "cheap vegetarian italian restaurant in rome"

test_output = retrieve_and_rerank(
    query=test_query,
    df_in=df,
    embeddings_array=embeddings,
    retrieval_top_k=100,
    rerank_top_k=5,
    verbose=False,
)

rag_context = build_rag_context(test_output["results"], top_n=5)

print("Allowed restaurant names:")
print(rag_context["allowed_restaurant_names"])

print("\nRAG context:")
print(rag_context["context_text"])

Allowed restaurant names:
['Sfiziarte - The Art of Food', 'Pizza & Friends', 'Bacio Di Puglia', 'Sto Bene Roma', 'Ristorcaffe Vecchia Roma']

RAG context:
[1] Sfiziarte - The Art of Food
Location: Rome, Italy
Rating: 5.0/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Mediterranean, European
Special diets: Vegetarian Friendly, Vegan Options, Gluten Free Options
Meals: Breakfast, Lunch, Dinner, Brunch
Popularity: #27 of 10231 Restaurants in Rome
Matched constraints: city=rome, diet=vegetarian friendly, price=cheap, tag=italian

[2] Pizza & Friends
Location: Rome, Italy
Rating: 5.0/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Pizza, Vegetarian Friendly
Special diets: Vegetarian Friendly, Vegan Options
Meals: Lunch, Dinner
Popularity: #1582 of 10232 Restaurants in Rome
Matched constraints: city=rome, diet=vegetarian friendly, price=cheap, tag=italian

[3] Bacio Di Puglia
Location: Rome, Italy
Rating: 4.5/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Fast

In [ ]:
def describe_match_from_evidence(evidence, parsed_query):
    reasons = []

    price = parsed_query.get("price_bucket")
    tags = parsed_query.get("tags", [])
    dietary = parsed_query.get("dietary", [])
    meals = parsed_query.get("matched_meals", [])
    features = parsed_query.get("matched_features", [])

    evidence_price = evidence.get("price_bucket", "unknown")
    evidence_tags = normalize_text(evidence.get("cuisines_and_tags", ""))
    evidence_diets = normalize_text(evidence.get("special_diets", ""))
    evidence_meals = normalize_text(evidence.get("meals", ""))
    evidence_features = normalize_text(evidence.get("features", ""))

    if price and evidence_price == price:
        reasons.append(f"it matches the requested budget ({price})")

    for tag in tags:
        if contains_phrase(evidence_tags, normalize_text(tag)):
            reasons.append(f"it has a relevant cuisine/tag: {tag}")

    for diet in dietary:
        diet_norm = normalize_text(diet)
        if (
            contains_phrase(evidence_diets, diet_norm)
            or contains_phrase(evidence_tags, diet_norm)
            or contains_phrase(evidence_features, diet_norm)
        ):
            reasons.append(f"it supports the requested dietary preference: {diet}")

    for meal in meals:
        meal_norm = normalize_text(meal)
        if contains_phrase(evidence_meals, meal_norm) or contains_phrase(evidence_tags, meal_norm):
            reasons.append(f"it is suitable for {meal}")

    for feature in features:
        feature_norm = normalize_text(feature)
        if contains_phrase(evidence_features, feature_norm) or contains_phrase(evidence_tags, feature_norm):
            reasons.append(f"it has a relevant feature: {feature}")

    if not reasons:
        if evidence.get("rating", "unknown") != "unknown":
            reasons.append(f"it has a rating of {evidence.get('rating')}")
        else:
            reasons.append("it is one of the most relevant restaurants retrieved from the dataset")

    return reasons[:3]


def generate_template_answer(query, parsed_query, rag_context, max_recommendations=3):
    restaurants = rag_context["restaurants"][:max_recommendations]

    if len(restaurants) == 0:
        city = parsed_query.get("city")
        country = parsed_query.get("country")

        if city:
            return (
                f"I could not find matching restaurants in the dataset for the city “{city}”. "
                "You can try a nearby city or a broader request."
            )

        if country:
            return (
                f"I could not find matching restaurants in the dataset for “{country}”. "
                "You can try relaxing some constraints."
            )

        return (
            "I could not find sufficiently relevant restaurants in the dataset for this request. "
            "Try specifying a city, cuisine, or budget."
        )

    intro = "Here are a few relevant options based on the restaurants retrieved from the dataset:\n"
    lines = [intro]

    for i, evidence in enumerate(restaurants, start=1):
        name = evidence["name"]
        location = evidence["location"]
        rating = evidence["rating"]
        price = evidence["price_bucket"]

        reasons = describe_match_from_evidence(evidence, parsed_query)

        line = (
            f"{i}. **{name}** — {location}. "
            f"Rating: {rating}; price: {price}. "
            f"Why it matches: " + "; ".join(reasons) + "."
        )

        lines.append(line)

    lines.append(
        "\nThis recommendation is based only on the retrieved restaurants and their available metadata."
    )

    return "\n".join(lines)

In [ ]:
template_answer = generate_template_answer(
    query=test_query,
    parsed_query=test_output["parsed_query"],
    rag_context=rag_context,
    max_recommendations=3,
)

print(template_answer)

Here are a few relevant options based on the restaurants retrieved from the dataset:

1. **Sfiziarte - The Art of Food** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.
2. **Pizza & Friends** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.
3. **Bacio Di Puglia** — Rome, Italy. Rating: 4.5/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.

This recommendation is based only on the retrieved restaurants and their available metadata.


In [ ]:
def build_llm_prompt(query, parsed_query, rag_context, max_recommendations=3):
    allowed_names = rag_context["allowed_restaurant_names"][:max_recommendations]
    context_text = rag_context["context_text"]

    prompt = f"""
You are TableWise, a restaurant recommendation assistant.

User query:
{query}

Parsed query constraints:
{json.dumps(parsed_query, ensure_ascii=False, indent=2)}

Retrieved restaurant evidence:
{context_text}

Instructions:
- Recommend at most {max_recommendations} restaurants.
- Use ONLY restaurants listed in the retrieved evidence.
- Do NOT invent restaurant names.
- Do NOT mention restaurants outside this list: {allowed_names}
- For each recommendation, briefly explain why it matches the query.
- If there are no restaurants in the evidence, say that no matching restaurants were found.
- Answer in English.
- Keep the answer concise.

Final answer:
""".strip()

    return prompt

In [ ]:
prompt = build_llm_prompt(
    query=test_query,
    parsed_query=test_output["parsed_query"],
    rag_context=rag_context,
    max_recommendations=3,
)

print(prompt[:4000])

You are TableWise, a restaurant recommendation assistant.

User query:
cheap vegetarian italian restaurant in rome

Parsed query constraints:
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Retrieved restaurant evidence:
[1] Sfiziarte - The Art of Food
Location: Rome, Italy
Rating: 5.0/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italian, Mediterranean, European
Special diets: Vegetarian Friendly, Vegan Options, Gluten Free Options
Meals: Breakfast, Lunch, Dinner, Brunch
Popularity: #27 of 10231 Restaurants in Rome
Matched constraints: city=rome, diet=vegetarian friendly, price=cheap, tag=italian

[2] Pizza & Friends
Location: Rome, Italy
Rating: 5.0/5
Price bucket: cheap
Cuisines/tags: Cheap Eats, Italia

In [ ]:
USE_HF_GENERATOR = False

if USE_HF_GENERATOR:
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

    HF_GENERATION_MODEL_NAME = "google/flan-t5-large"

    print("Loading HF generation model:", HF_GENERATION_MODEL_NAME)

    tokenizer = AutoTokenizer.from_pretrained(HF_GENERATION_MODEL_NAME)
    generation_model = AutoModelForSeq2SeqLM.from_pretrained(
        HF_GENERATION_MODEL_NAME,
        device_map="auto" if torch.cuda.is_available() else None,
    )

    hf_generator = pipeline(
        "text2text-generation",
        model=generation_model,
        tokenizer=tokenizer,
        max_new_tokens=350,
        do_sample=False,
    )

    print("HF generator loaded.")
else:
    print("HF generator disabled. Using template answer by default.")

HF generator disabled. Using template answer by default.


In [ ]:
def generate_hf_answer(prompt):
    if not USE_HF_GENERATOR:
        return None

    output = hf_generator(prompt)[0]["generated_text"]
    return output.strip()

In [ ]:
if USE_HF_GENERATOR:
    llm_answer = generate_hf_answer(prompt)
    print(llm_answer)
else:
    llm_answer = None
    print("Skipped HF generation.")

Skipped HF generation.


In [ ]:
def normalize_name_for_check(name):
    return normalize_text(name)


def extract_allowed_name_mentions(answer, allowed_names):
    answer_norm = normalize_text(answer)

    mentions = []

    for name in allowed_names:
        name_norm = normalize_name_for_check(name)

        if name_norm and name_norm in answer_norm:
            mentions.append(name)

    return mentions


def find_potential_restaurant_name_lines(answer):
    """
    Heuristic simplu:
    - extrage bucăți bold **Name**
    - extrage începuturi de listă: "1. Name — ..."
    """
    candidates = []

    for m in re.finditer(r"\*\*(.*?)\*\*", answer):
        val = m.group(1).strip()
        if val:
            candidates.append(val)

    for line in answer.splitlines():
        line = line.strip()
        m = re.match(r"^\d+\.\s+(.+?)(?:\s+—|\s+-|\s*:|$)", line)
        if m:
            val = m.group(1).strip()
            if val:
                candidates.append(val)

    cleaned = []
    seen = set()

    for c in candidates:
        c_norm = normalize_text(c)
        if c_norm and c_norm not in seen:
            cleaned.append(c)
            seen.add(c_norm)

    return cleaned


def check_groundedness(answer, allowed_names):
    allowed_norms = {normalize_name_for_check(n): n for n in allowed_names}

    mentioned_candidates = find_potential_restaurant_name_lines(answer)

    unsupported = []

    for candidate in mentioned_candidates:
        cand_norm = normalize_name_for_check(candidate)

        if cand_norm not in allowed_norms:
            is_supported = any(cand_norm in allowed or allowed in cand_norm for allowed in allowed_norms.keys())

            if not is_supported:
                unsupported.append(candidate)

    supported_mentions = extract_allowed_name_mentions(answer, allowed_names)

    groundedness = {
        "allowed_names": allowed_names,
        "supported_mentions": supported_mentions,
        "potential_name_mentions": mentioned_candidates,
        "unsupported_mentions": unsupported,
        "is_grounded": len(unsupported) == 0,
        "groundedness_rate": None,
    }

    if len(mentioned_candidates) > 0:
        groundedness["groundedness_rate"] = (
            (len(mentioned_candidates) - len(unsupported)) / len(mentioned_candidates)
        )

    return groundedness

In [ ]:
groundedness_template = check_groundedness(
    template_answer,
    rag_context["allowed_restaurant_names"],
)

print(json.dumps(groundedness_template, indent=2, ensure_ascii=False))

{
  "allowed_names": [
    "Sfiziarte - The Art of Food",
    "Pizza & Friends",
    "Bacio Di Puglia",
    "Sto Bene Roma",
    "Ristorcaffe Vecchia Roma"
  ],
  "supported_mentions": [
    "Sfiziarte - The Art of Food",
    "Pizza & Friends",
    "Bacio Di Puglia"
  ],
  "potential_name_mentions": [
    "Sfiziarte - The Art of Food",
    "Pizza & Friends",
    "Bacio Di Puglia",
    "**Sfiziarte"
  ],
  "unsupported_mentions": [],
  "is_grounded": true,
  "groundedness_rate": 1.0
}


In [ ]:
def generate_safe_rag_answer(
    query,
    top_k=5,
    max_recommendations=3,
    use_llm=False,
    verbose=True,
):
    retrieval_output = retrieve_and_rerank(
        query=query,
        df_in=df,
        embeddings_array=embeddings,
        retrieval_top_k=100,
        rerank_top_k=top_k,
        verbose=verbose,
    )

    parsed_query = retrieval_output["parsed_query"]
    results = retrieval_output["results"]

    rag_context = build_rag_context(results, top_n=top_k)

    template_answer = generate_template_answer(
        query=query,
        parsed_query=parsed_query,
        rag_context=rag_context,
        max_recommendations=max_recommendations,
    )

    chosen_answer = template_answer
    generation_method = "template"

    llm_answer = None
    llm_groundedness = None

    if use_llm:
        prompt = build_llm_prompt(
            query=query,
            parsed_query=parsed_query,
            rag_context=rag_context,
            max_recommendations=max_recommendations,
        )

        llm_answer = generate_hf_answer(prompt)

        if llm_answer:
            llm_groundedness = check_groundedness(
                llm_answer,
                rag_context["allowed_restaurant_names"],
            )

            if llm_groundedness["is_grounded"]:
                chosen_answer = llm_answer
                generation_method = "llm"
            else:
                generation_method = "template_fallback_due_to_ungrounded_llm"

    final_groundedness = check_groundedness(
        chosen_answer,
        rag_context["allowed_restaurant_names"],
    )

    return {
        "query": query,
        "parsed_query": parsed_query,
        "retrieval_output": retrieval_output,
        "rag_context": rag_context,
        "answer": chosen_answer,
        "generation_method": generation_method,
        "template_answer": template_answer,
        "llm_answer": llm_answer,
        "llm_groundedness": llm_groundedness,
        "final_groundedness": final_groundedness,
    }

In [ ]:
query = "cheap vegetarian italian restaurant in rome"

rag_output = generate_safe_rag_answer(
    query=query,
    top_k=5,
    max_recommendations=3,
    use_llm=False,
    verbose=True,
)

print("Generation method:", rag_output["generation_method"])
print("\nAnswer:\n")
print(rag_output["answer"])

print("\nGroundedness:")
print(json.dumps(rag_output["final_groundedness"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Metadata score > 0 filter: 12603 -> 10182
After metadata filter/scoring: 10182
Computing semantic scores for 10,182 candidates...
Generation method: template

Answer:

Here are a few relevant options based on the restaurants retrieved from the dataset:

1. **Sfiziarte - The Art of Food** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.
2. **Pizza & Friends** — Rome, Italy. Rating: 5.0/5; price: cheap. W

In [ ]:
query = "expensive fine dining in milan"

rag_output = generate_safe_rag_answer(
    query=query,
    top_k=5,
    max_recommendations=3,
    use_llm=False,
    verbose=True,
)

print("Generation method:", rag_output["generation_method"])
print("\nAnswer:\n")
print(rag_output["answer"])

print("\nGroundedness:")
print(json.dumps(rag_output["final_groundedness"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "expensive fine dining in milan",
  "normalized_query": "expensive fine dining in milan",
  "city": "milan",
  "country": null,
  "price_bucket": "expensive",
  "tags": [
    "fine dining"
  ],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=milan: 1033798 -> 8381
After location filter: 8381
Metadata score > 0 filter: 8381 -> 2012
After metadata filter/scoring: 2012
Computing semantic scores for 2,012 candidates...
Generation method: template

Answer:

Here are a few relevant options based on the restaurants retrieved from the dataset:

1. **Seta** — Milan, Italy. Rating: 4.5/5; price: expensive. Why it matches: it matches the requested budget (expensive); it has a relevant cuisine/tag: fine dining.
2. **Ba Restaurant** — Milan, Italy. Rating: 4.5/5; price: expensive. Why it matches: it matches the requested budget (expensive); it has a relevant cuisine/tag: fine dining.
3. **LUME** — Milan, 

In [ ]:
query = "vegetarian brunch in barcelona"

rag_output = generate_safe_rag_answer(
    query=query,
    top_k=5,
    max_recommendations=3,
    use_llm=False,
    verbose=True,
)

print("Generation method:", rag_output["generation_method"])
print("\nAnswer:\n")
print(rag_output["answer"])

print("\nGroundedness:")
print(json.dumps(rag_output["final_groundedness"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "vegetarian brunch in barcelona",
  "normalized_query": "vegetarian brunch in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [
    "brunch"
  ],
  "matched_features": []
}

Initial rows: 1033798
City filter city=barcelona: 1033798 -> 10283
After location filter: 10283
Metadata score > 0 filter: 10283 -> 3713
After metadata filter/scoring: 3713
Computing semantic scores for 3,713 candidates...
Generation method: template

Answer:

Here are a few relevant options based on the restaurants retrieved from the dataset:

1. **Eixampeling Brunch Café & Bar** — Barcelona, Spain. Rating: 5.0/5; price: mid. Why it matches: it supports the requested dietary preference: vegetarian friendly; it is suitable for brunch.
2. **Ugot Bruncherie** — Barcelona, Spain. Rating: 4.5/5; price: expensive. Why it matches: it supports the requested dietary preference: ve

In [ ]:
query = "vegetarian restaurant in berlin"

rag_output = generate_safe_rag_answer(
    query=query,
    top_k=5,
    max_recommendations=3,
    use_llm=False,
    verbose=True,
)

print("Generation method:", rag_output["generation_method"])
print("\nAnswer:\n")
print(rag_output["answer"])

print("\nGroundedness:")
print(json.dumps(rag_output["final_groundedness"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "vegetarian restaurant in berlin",
  "normalized_query": "vegetarian restaurant in berlin",
  "city": "berlin",
  "country": null,
  "price_bucket": null,
  "tags": [],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=berlin: 1033798 -> 0
Oraș detectat, dar fără rezultate în dataset. Returnăm subset gol.
After location filter: 0
Generation method: template

Answer:

I could not find matching restaurants in the dataset for the city “berlin”. You can try a nearby city or a broader request.

Groundedness:
{
  "allowed_names": [],
  "supported_mentions": [],
  "potential_name_mentions": [],
  "unsupported_mentions": [],
  "is_grounded": true,
  "groundedness_rate": null
}


In [ ]:
test_queries = [
    "cheap italian restaurant in rome",
    "vegetarian brunch in barcelona",
    "romantic dinner in paris",
    "family friendly seafood place in lisbon",
    "coffee and breakfast in athens",
    "expensive fine dining in milan",
    "vegan restaurant near berlin with outdoor seating",
    "tapas place in madrid",
    "family restaurant in london",
    "gluten free restaurant in paris",
]

rag_batch_outputs = {}
rag_batch_rows = []

for q in test_queries:
    print("=" * 120)
    print("Running:", q)

    out = generate_safe_rag_answer(
        query=q,
        top_k=5,
        max_recommendations=3,
        use_llm=False,
        verbose=False,
    )

    rag_batch_outputs[q] = out

    results = out["retrieval_output"]["results"]
    groundedness = out["final_groundedness"]

    rag_batch_rows.append({
        "query": q,
        "parsed_city": out["parsed_query"].get("city"),
        "parsed_country": out["parsed_query"].get("country"),
        "parsed_price_bucket": out["parsed_query"].get("price_bucket"),
        "num_results": len(results),
        "generation_method": out["generation_method"],
        "is_grounded": groundedness["is_grounded"],
        "groundedness_rate": groundedness["groundedness_rate"],
        "unsupported_mentions": groundedness["unsupported_mentions"],
        "answer_preview": out["answer"][:300],
    })

rag_batch_df = pd.DataFrame(rag_batch_rows)
display(rag_batch_df)

Running: cheap italian restaurant in rome
Running: vegetarian brunch in barcelona
Running: romantic dinner in paris
Running: family friendly seafood place in lisbon
Running: coffee and breakfast in athens
Running: expensive fine dining in milan
Running: vegan restaurant near berlin with outdoor seating
Running: tapas place in madrid
Running: family restaurant in london
Running: gluten free restaurant in paris


,query,parsed_city,parsed_country,parsed_price_bucket,num_results,generation_method,is_grounded,groundedness_rate,unsupported_mentions,answer_preview
0,cheap italian restaurant in rome,rome,None,cheap,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Pizza & Friends** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian.\n2. **Stefano of Rome** — Rome, Italy. Rating: 5.0/"
1,vegetarian brunch in barcelona,barcelona,None,None,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Eixampeling Brunch Café & Bar** — Barcelona, Spain. Rating: 5.0/5; price: mid. Why it matches: it supports the requested dietary preference: vegetarian friendly; it is suitable for brunch.\n2. **Ugot Bruncherie"
2,romantic dinner in paris,paris,None,None,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Jean Yves of Chef Jean-Yves' Table** — Paris, France. Rating: 5.0/5; price: expensive. Why it matches: it is suitable for dinner.\n2. **Restaurant de la Tour** — Paris, France. Rating: 4.5/5; price: expensive."
3,family friendly seafood place in lisbon,lisbon,None,None,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Solar 31 - Peixe e Marisco** — Lisbon, Portugal. Rating: 5.0/5; price: expensive. Why it matches: it has a relevant cuisine/tag: seafood.\n2. **A Taberna do Mar** — Lisbon, Portugal. Rating: 5.0/5; price: expen"
4,coffee and breakfast in athens,athens,None,None,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Coffee Joint** — Athens, Greece. Rating: 5.0/5; price: cheap. Why it matches: it is suitable for breakfast.\n2. **Victory Cafe** — Athens, Greece. Rating: 5.0/5; price: mid. Why it matches: it is suitable for b"
5,expensive fine dining in milan,milan,None,expensive,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Seta** — Milan, Italy. Rating: 4.5/5; price: expensive. Why it matches: it matches the requested budget (expensive); it has a relevant cuisine/tag: fine dining.\n2. **Ba Restaurant** — Milan, Italy. Rating: 4.5"
6,vegan restaurant near berlin with outdoor seating,berlin,None,None,0,template,True,NaN,[],I could not find matching restaurants in the dataset for the city “berlin”. You can try a nearby city or a broader request.
7,tapas place in madrid,madrid,None,None,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Tapa Tapa Arenal** — Madrid, Spain. Rating: 4.5/5; price: expensive. Why it matches: it has a relevant cuisine/tag: spanish.\n2. **Tinto y Tapas** — Madrid, Spain. Rating: 4.5/5; price: mid. Why it matches: it"
8,family restaurant in london,london,None,None,0,template,True,NaN,[],I could not find matching restaurants in the dataset for the city “london”. You can try a nearby city or a broader request.
9,gluten free restaurant in paris,paris,None,None,5,template,True,1.0,[],"Here are a few relevant options based on the restaurants retrieved from the dataset:\n\n1. **Restaurant H** — Paris, France. Rating: 5.0/5; price: expensive. Why it matches: it supports the requested dietary preference: gluten free options.\n2. **Pur' - Jean-François Rouquette** — Paris, France. Rating"


In [ ]:
for q, out in rag_batch_outputs.items():
    print("=" * 120)
    print("QUERY:", q)
    print("PARSED:", json.dumps(out["parsed_query"], ensure_ascii=False))
    print("METHOD:", out["generation_method"])
    print("ANSWER:")
    print(out["answer"])
    print("GROUNDED:", out["final_groundedness"]["is_grounded"])

QUERY: cheap italian restaurant in rome
PARSED: {"original_query": "cheap italian restaurant in rome", "normalized_query": "cheap italian restaurant in rome", "city": "rome", "country": null, "price_bucket": "cheap", "tags": ["italian"], "dietary": [], "matched_meals": [], "matched_features": []}
METHOD: template
ANSWER:
Here are a few relevant options based on the restaurants retrieved from the dataset:

1. **Pizza & Friends** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian.
2. **Stefano of Rome** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian.
3. **Ristorante Pizzeria Andrea** — Rome, Italy. Rating: 4.5/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian.

This recommendation is based only on the retrieved restaurants and their available meta

In [ ]:
rag_metrics = {
    "num_test_queries": len(rag_batch_df),
    "grounded_answer_rate": float(rag_batch_df["is_grounded"].mean()),
    "num_ungrounded_answers": int((~rag_batch_df["is_grounded"]).sum()),
    "mean_groundedness_rate": float(rag_batch_df["groundedness_rate"].dropna().mean()) if rag_batch_df["groundedness_rate"].notna().any() else None,
    "template_generation_rate": float((rag_batch_df["generation_method"] == "template").mean()),
    "no_result_queries": int((rag_batch_df["num_results"] == 0).sum()),
}

rag_metrics_df = pd.DataFrame([rag_metrics])
display(rag_metrics_df)

,num_test_queries,grounded_answer_rate,num_ungrounded_answers,mean_groundedness_rate,template_generation_rate,no_result_queries
0,10,1.0,0,1.0,1.0,2


In [ ]:
def answer_mentions_constraint(answer, constraint):
    if not constraint:
        return None

    answer_norm = normalize_text(answer)
    constraint_norm = normalize_text(constraint)

    return contains_phrase(answer_norm, constraint_norm)


def compute_answer_constraint_coverage(out):
    parsed = out["parsed_query"]
    answer = out["answer"]

    constraints = []

    if parsed.get("city"):
        constraints.append(parsed["city"])

    if parsed.get("country"):
        constraints.append(parsed["country"])

    if parsed.get("price_bucket"):
        constraints.append(parsed["price_bucket"])

    constraints.extend(parsed.get("tags", []))
    constraints.extend(parsed.get("dietary", []))
    constraints.extend(parsed.get("matched_meals", []))
    constraints.extend(parsed.get("matched_features", []))

    if not constraints:
        return None

    hits = []

    for c in constraints:
        hit = answer_mentions_constraint(answer, c)
        if hit is not None:
            hits.append(hit)

    if not hits:
        return None

    return float(np.mean(hits))


coverage_rows = []

for q, out in rag_batch_outputs.items():
    coverage_rows.append({
        "query": q,
        "constraint_coverage_in_answer": compute_answer_constraint_coverage(out),
    })

coverage_df = pd.DataFrame(coverage_rows)
display(coverage_df)

print("Mean constraint coverage:", coverage_df["constraint_coverage_in_answer"].dropna().mean())

,query,constraint_coverage_in_answer
0,cheap italian restaurant in rome,1.000000
1,vegetarian brunch in barcelona,1.000000
2,romantic dinner in paris,0.666667
3,family friendly seafood place in lisbon,0.666667
4,coffee and breakfast in athens,1.000000
5,expensive fine dining in milan,1.000000
6,vegan restaurant near berlin with outdoor seating,0.250000
7,tapas place in madrid,1.000000
8,family restaurant in london,1.000000
9,gluten free restaurant in paris,1.000000


Mean constraint coverage: 0.8583333333333332


In [ ]:
RAG_BATCH_SUMMARY_PATH = OUTPUT_DIR / "rag_batch_summary.csv"
RAG_METRICS_PATH = OUTPUT_DIR / "rag_metrics.csv"
RAG_CONSTRAINT_COVERAGE_PATH = OUTPUT_DIR / "rag_constraint_coverage.csv"

rag_batch_df.to_csv(RAG_BATCH_SUMMARY_PATH, index=False)
rag_metrics_df.to_csv(RAG_METRICS_PATH, index=False)
coverage_df.to_csv(RAG_CONSTRAINT_COVERAGE_PATH, index=False)

print("Saved:")
print("-", RAG_BATCH_SUMMARY_PATH)
print("-", RAG_METRICS_PATH)
print("-", RAG_CONSTRAINT_COVERAGE_PATH)

Saved:
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/rag_batch_summary.csv
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/rag_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/rag_constraint_coverage.csv


In [ ]:
RAG_ANSWERS_PATH = OUTPUT_DIR / "rag_answers_examples.json"

answers_payload = []

for q, out in rag_batch_outputs.items():
    answers_payload.append({
        "query": q,
        "parsed_query": out["parsed_query"],
        "generation_method": out["generation_method"],
        "answer": out["answer"],
        "allowed_restaurant_names": out["rag_context"]["allowed_restaurant_names"],
        "groundedness": out["final_groundedness"],
        "num_results": len(out["retrieval_output"]["results"]),
    })

with open(RAG_ANSWERS_PATH, "w", encoding="utf-8") as f:
    json.dump(answers_payload, f, indent=2, ensure_ascii=False)

print("Saved:", RAG_ANSWERS_PATH)

Saved: /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/rag_answers_examples.json


In [ ]:
rag_config = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "faiss_dir": str(FAISS_DIR),
    "mapping_path": str(MAPPING_PATH),
    "embeddings_path": str(EMBEDDINGS_PATH),
    "embedding_model": EMBEDDING_MODEL_NAME,
    "generation_policy": {
        "default_method": "template",
        "optional_llm_enabled": bool(USE_HF_GENERATOR),
        "max_recommendations": 3,
        "groundedness_checker": "restaurant names in final answer must be in allowed retrieved restaurant names",
        "fallback_policy": "use template answer if LLM answer is ungrounded",
    },
    "rag_metrics": rag_metrics,
}

RAG_CONFIG_PATH = OUTPUT_DIR / "rag_config.json"

with open(RAG_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(rag_config, f, indent=2, ensure_ascii=False)

print("Saved:", RAG_CONFIG_PATH)
rag_config

Saved: /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/rag_config.json


{'created_at': '2026-05-04T17:32:11.754344Z',
 'faiss_dir': '/content/drive/MyDrive/tablewise/artifacts_new/faiss',
 'mapping_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_index_mapping.parquet',
 'embeddings_path': '/content/drive/MyDrive/tablewise/artifacts_new/faiss/restaurant_embeddings.npy',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'generation_policy': {'default_method': 'template',
  'optional_llm_enabled': False,
  'max_recommendations': 3,
  'groundedness_checker': 'restaurant names in final answer must be in allowed retrieved restaurant names',
  'fallback_policy': 'use template answer if LLM answer is ungrounded'},
 'rag_metrics': {'num_test_queries': 10,
  'grounded_answer_rate': 1.0,
  'num_ungrounded_answers': 0,
  'mean_groundedness_rate': 1.0,
  'template_generation_rate': 1.0,
  'no_result_queries': 2}}

In [ ]:
def tablewise_answer(
    query,
    top_k=5,
    max_recommendations=3,
    use_llm=False,
    verbose=False,
):

    return generate_safe_rag_answer(
        query=query,
        top_k=top_k,
        max_recommendations=max_recommendations,
        use_llm=use_llm,
        verbose=verbose,
    )

In [ ]:
demo_queries = [
    "cheap vegetarian italian restaurant in rome",
    "expensive fine dining in milan",
    "vegetarian brunch in barcelona",
    "tapas place in madrid",
    "family restaurant in london",
]

for q in demo_queries:
    out = tablewise_answer(
        q,
        top_k=5,
        max_recommendations=3,
        use_llm=False,
        verbose=False,
    )

    print("=" * 120)
    print("USER:", q)
    print("TABLEWISE:")
    print(out["answer"])
    print("Grounded:", out["final_groundedness"]["is_grounded"])

USER: cheap vegetarian italian restaurant in rome
TABLEWISE:
Here are a few relevant options based on the restaurants retrieved from the dataset:

1. **Sfiziarte - The Art of Food** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.
2. **Pizza & Friends** — Rome, Italy. Rating: 5.0/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.
3. **Bacio Di Puglia** — Rome, Italy. Rating: 4.5/5; price: cheap. Why it matches: it matches the requested budget (cheap); it has a relevant cuisine/tag: italian; it supports the requested dietary preference: vegetarian friendly.

This recommendation is based only on the retrieved restaurants and their available metadata.
Grounded: True
USER: expensive fine dining in milan
TA

## Testing LLM generation

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.5 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

USE_HF_GENERATOR = True

HF_GENERATION_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

HF_MAX_NEW_TOKENS = 450
HF_TEMPERATURE = 0.1

print("USE_HF_GENERATOR:", USE_HF_GENERATOR)
print("HF model:", HF_GENERATION_MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

USE_HF_GENERATOR: True
HF model: Qwen/Qwen2.5-1.5B-Instruct
CUDA available: True
GPU: NVIDIA L4


In [ ]:
# HF_GENERATION_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
if USE_HF_GENERATOR:
    hf_tokenizer = AutoTokenizer.from_pretrained(
        HF_GENERATION_MODEL_NAME,
        trust_remote_code=True,
    )

    hf_model = AutoModelForCausalLM.from_pretrained(
        HF_GENERATION_MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )

    hf_model.eval()

    print("HF generation model loaded.")
else:
    hf_tokenizer = None
    hf_model = None
    print("HF generator disabled.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

HF generation model loaded.


In [ ]:
def build_strict_hf_prompt(query, parsed_query, rag_context, max_recommendations=3):
    allowed_names = rag_context["allowed_restaurant_names"][:max_recommendations]
    context_text = rag_context["context_text"]

    style_instructions = [
        "Write like a helpful restaurant assistant, not like a database report.",
        "Avoid repeating the same sentence pattern for every restaurant.",
        "Do not start every answer with 'Based on your request' or 'Based on your criteria'.",
        "Use natural transitions, but keep the answer concise.",
        "Do not use marketing claims such as 'popular among locals and tourists', 'excellent service', 'upscale atmosphere', 'cozy atmosphere', or 'perfect choice' unless those exact ideas are supported by the evidence.",
        "Do not say a restaurant is vegan-friendly unless the evidence explicitly says Vegan Options.",
        "Do not say gluten-free unless the evidence explicitly says Gluten Free Options.",
        "Do not say family-friendly, romantic, outdoor seating, or wheelchair accessible unless the evidence explicitly supports it.",
        "Mention uncertainty when evidence is limited.",
    ]

    if len(allowed_names) == 0:
        user_content = f"""
User query:
{query}

Parsed query constraints:
{json.dumps(parsed_query, ensure_ascii=False, indent=2)}

Retrieved restaurant evidence:
No restaurants were retrieved.

Instructions:
- Say that no matching restaurants were found in the dataset.
- Do not recommend any restaurant.
- Answer in English.
- Keep it short and natural.
""".strip()
    else:
        user_content = f"""
User query:
{query}

Parsed query constraints:
{json.dumps(parsed_query, ensure_ascii=False, indent=2)}

Allowed restaurant names:
{json.dumps(allowed_names, ensure_ascii=False, indent=2)}

Retrieved restaurant evidence:
{context_text}

Strict grounding rules:
- Use ONLY the retrieved restaurant evidence.
- Mention ONLY restaurant names from the allowed restaurant names list.
- Recommend at most {max_recommendations} restaurants.
- Do NOT invent restaurant names.
- Do NOT invent facts, atmosphere, service quality, popularity, location details, or dietary attributes.
- If the evidence only says "Vegetarian Friendly", say vegetarian-friendly only. Do not infer vegan.
- If a field is unknown, do not mention it.
- If the evidence is limited, say "from the available metadata" or avoid the claim.

Writing style:
{chr(10).join("- " + s for s in style_instructions)}

Output requirements:
- Answer in English.
- Keep the answer under 160 words.
- Use either a short paragraph or a compact list.
- Do not include a final generic sentence like "Enjoy your meal" every time.
- Do not use markdown tables.

Final answer:
""".strip()

    messages = [
        {
            "role": "system",
            "content": (
                "You are TableWise, a grounded restaurant recommendation assistant. "
                "You must only use the retrieved evidence. "
                "You must not invent unsupported attributes. "
                "You may vary wording naturally, but restaurant names and facts must stay grounded."
            ),
        },
        {
            "role": "user",
            "content": user_content,
        },
    ]

    return messages

In [ ]:
def get_model_device(model):
    try:
        return next(model.parameters()).device
    except Exception:
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def generate_hf_chat_answer(
    messages,
    max_new_tokens=HF_MAX_NEW_TOKENS,
    temperature=0.45,
    top_p=0.9,
    creative=True,
):
    if not USE_HF_GENERATOR:
        return None

    model_device = get_model_device(hf_model)

    if hasattr(hf_tokenizer, "apply_chat_template"):
        try:
            prompt_text = hf_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            prompt_text = "\n\n".join(
                [f"{m['role'].upper()}: {m['content']}" for m in messages]
            )
            prompt_text += "\n\nASSISTANT:"
    else:
        prompt_text = "\n\n".join(
            [f"{m['role'].upper()}: {m['content']}" for m in messages]
        )
        prompt_text += "\n\nASSISTANT:"

    inputs = hf_tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=4096,
    )

    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    if creative:
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": True,
            "temperature": temperature,
            "top_p": top_p,
            "repetition_penalty": 1.08,
            "pad_token_id": hf_tokenizer.eos_token_id,
            "eos_token_id": hf_tokenizer.eos_token_id,
        }
    else:
        generation_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": False,
            "repetition_penalty": 1.05,
            "pad_token_id": hf_tokenizer.eos_token_id,
            "eos_token_id": hf_tokenizer.eos_token_id,
        }

    with torch.no_grad():
        generated_ids = hf_model.generate(
            **inputs,
            **generation_kwargs,
        )

    input_length = inputs["input_ids"].shape[-1]
    new_tokens = generated_ids[0][input_length:]

    answer = hf_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return answer

In [ ]:
def generate_safe_hf_rag_answer(
    query,
    top_k=5,
    max_recommendations=3,
    verbose=True,
):
    retrieval_output = retrieve_and_rerank(
        query=query,
        df_in=df,
        embeddings_array=embeddings,
        retrieval_top_k=100,
        rerank_top_k=top_k,
        verbose=verbose,
    )

    parsed_query = retrieval_output["parsed_query"]
    results = retrieval_output["results"]

    rag_context = build_rag_context(results, top_n=top_k)

    template_answer = generate_template_answer(
        query=query,
        parsed_query=parsed_query,
        rag_context=rag_context,
        max_recommendations=max_recommendations,
    )

    template_groundedness = check_groundedness(
        template_answer,
        rag_context["allowed_restaurant_names"],
    )

    if len(rag_context["restaurants"]) == 0:
        return {
            "query": query,
            "parsed_query": parsed_query,
            "retrieval_output": retrieval_output,
            "rag_context": rag_context,
            "messages": None,
            "llm_answer": None,
            "template_answer": template_answer,
            "answer": template_answer,
            "generation_method": "template_no_results",
            "llm_error": None,
            "llm_groundedness": None,
            "template_groundedness": template_groundedness,
            "final_groundedness": template_groundedness,
        }

    messages = build_strict_hf_prompt(
        query=query,
        parsed_query=parsed_query,
        rag_context=rag_context,
        max_recommendations=max_recommendations,
    )

    llm_answer = None
    llm_error = None
    llm_groundedness = None

    try:
        llm_answer = generate_hf_chat_answer(messages)

        llm_groundedness = check_groundedness(
            llm_answer,
            rag_context["allowed_restaurant_names"],
        )

        if llm_groundedness["is_grounded"]:
            final_answer = llm_answer
            generation_method = "hf_llm"
            final_groundedness = llm_groundedness
        else:
            final_answer = template_answer
            generation_method = "template_fallback_due_to_ungrounded_hf_llm"
            final_groundedness = template_groundedness

    except Exception as e:
        llm_error = repr(e)
        final_answer = template_answer
        generation_method = "template_fallback_due_to_hf_error"
        final_groundedness = template_groundedness

    return {
        "query": query,
        "parsed_query": parsed_query,
        "retrieval_output": retrieval_output,
        "rag_context": rag_context,
        "messages": messages,
        "llm_answer": llm_answer,
        "template_answer": template_answer,
        "answer": final_answer,
        "generation_method": generation_method,
        "llm_error": llm_error,
        "llm_groundedness": llm_groundedness,
        "template_groundedness": template_groundedness,
        "final_groundedness": final_groundedness,
    }

In [ ]:
hf_test_query = "cheap vegetarian italian restaurant in rome"

hf_test_output = generate_safe_hf_rag_answer(
    query=hf_test_query,
    top_k=5,
    max_recommendations=3,
    verbose=True,
)

print("Generation method:", hf_test_output["generation_method"])

if hf_test_output["llm_error"]:
    print("\nLLM error:")
    print(hf_test_output["llm_error"])

print("\nAllowed restaurant names:")
print(hf_test_output["rag_context"]["allowed_restaurant_names"])

print("\nHF LLM answer:")
print(hf_test_output["llm_answer"])

print("\nFinal answer used:")
print(hf_test_output["answer"])

print("\nHF groundedness:")
print(json.dumps(hf_test_output["llm_groundedness"], indent=2, ensure_ascii=False))

print("\nFinal groundedness:")
print(json.dumps(hf_test_output["final_groundedness"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "cheap vegetarian italian restaurant in rome",
  "normalized_query": "cheap vegetarian italian restaurant in rome",
  "city": "rome",
  "country": null,
  "price_bucket": "cheap",
  "tags": [
    "italian"
  ],
  "dietary": [
    "vegetarian friendly"
  ],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=rome: 1033798 -> 12603
After location filter: 12603
Metadata score > 0 filter: 12603 -> 10182
After metadata filter/scoring: 10182
Computing semantic scores for 10,182 candidates...
Generation method: hf_llm

Allowed restaurant names:
['Sfiziarte - The Art of Food', 'Pizza & Friends', 'Bacio Di Puglia', 'Sto Bene Roma', 'Ristorcaffe Vecchia Roma']

HF LLM answer:
Based on your query for a cheap vegetarian Italian restaurant in Rome, I recommend Sfiziarte - The Art of Food and Pizza & Friends. Both offer excellent vegetarian options within a budget, with Sfiziarte having a perfect rating and being highly popula

In [ ]:
hf_test_queries = [
    "cheap italian restaurant in rome",
    "cheap vegetarian italian restaurant in rome",
    "vegetarian brunch in barcelona",
    "romantic dinner in paris",
    "family friendly seafood place in lisbon",
    "coffee and breakfast in athens",
    "expensive fine dining in milan",
    "vegan restaurant near berlin with outdoor seating",
    "tapas place in madrid",
    "family restaurant in london",
    "gluten free restaurant in paris",
]

hf_batch_outputs = {}
hf_batch_rows = []

for q in hf_test_queries:
    print("=" * 120)
    print("Running:", q)

    out = generate_safe_hf_rag_answer(
        query=q,
        top_k=5,
        max_recommendations=3,
        verbose=False,
    )

    hf_batch_outputs[q] = out

    results = out["retrieval_output"]["results"]
    final_groundedness = out["final_groundedness"]
    llm_groundedness = out["llm_groundedness"]

    hf_batch_rows.append({
        "query": q,
        "parsed_city": out["parsed_query"].get("city"),
        "parsed_country": out["parsed_query"].get("country"),
        "parsed_price_bucket": out["parsed_query"].get("price_bucket"),
        "num_results": len(results),
        "generation_method": out["generation_method"],
        "llm_called": out["llm_answer"] is not None,
        "llm_error": out["llm_error"],
        "llm_is_grounded": None if llm_groundedness is None else llm_groundedness["is_grounded"],
        "llm_unsupported_mentions": None if llm_groundedness is None else llm_groundedness["unsupported_mentions"],
        "final_is_grounded": final_groundedness["is_grounded"],
        "final_groundedness_rate": final_groundedness["groundedness_rate"],
        "allowed_restaurant_names": out["rag_context"]["allowed_restaurant_names"],
        "final_answer_preview": out["answer"][:400],
    })

hf_batch_df = pd.DataFrame(hf_batch_rows)
display(hf_batch_df)

Running: cheap italian restaurant in rome
Running: cheap vegetarian italian restaurant in rome
Running: vegetarian brunch in barcelona
Running: romantic dinner in paris
Running: family friendly seafood place in lisbon
Running: coffee and breakfast in athens
Running: expensive fine dining in milan
Running: vegan restaurant near berlin with outdoor seating
Running: tapas place in madrid
Running: family restaurant in london
Running: gluten free restaurant in paris


,query,parsed_city,parsed_country,parsed_price_bucket,num_results,generation_method,llm_called,llm_error,llm_is_grounded,llm_unsupported_mentions,final_is_grounded,final_groundedness_rate,allowed_restaurant_names,final_answer_preview
0,cheap italian restaurant in rome,rome,None,cheap,5,hf_llm,True,None,True,[],True,1.0,"[Pizza & Friends, Stefano of Rome, Ristorante Pizzeria Andrea, Sto Bene Roma, New Bar]","Given your request for a cheap Italian restaurant in Rome, I recommend:\n\n1. **Pizza & Friends** - This highly-rated establishment offers an authentic Italian experience without breaking the bank. Known for its delicious pizzas and friendly service, it's perfect for a quick lunch or dinner.\n\n2. **Stefano of Rome** - Another top choice, this restaurant provides excellent value for money with..."
1,cheap vegetarian italian restaurant in rome,rome,None,cheap,5,hf_llm,True,None,True,[],True,1.0,"[Sfiziarte - The Art of Food, Pizza & Friends, Bacio Di Puglia, Sto Bene Roma, Ristorcaffe Vecchia Roma]","Based on your criteria for a cheap vegetarian Italian restaurant in Rome, I recommend:\n\n1. **Sfiziarte - The Art of Food** (Rating: 5.0/5) - This highly-rated establishment offers a variety of meals including breakfast, lunch, dinner, and brunch, making it perfect for all-day dining in Rome. It's known for its affordable prices and offers both vegetarian options and vegan choices.\n\n2. **Pi..."
2,vegetarian brunch in barcelona,barcelona,None,None,5,hf_llm,True,None,True,[],True,1.0,"[Eixampeling Brunch Café & Bar, Ugot Bruncherie, Bo Kaap, Bo de Boqueria, Carite Bar & Lounge]","Based on your request for a vegetarian brunch in Barcelona, I recommend:\n\n1. **Eixampeling Brunch Café & Bar** - This popular spot offers a delightful brunch experience with a rating of 5.0/5. It's located in Barcelona and is known for its mid-range prices, offering both breakfast and lunch options. The café also serves drinks and has been ranked as the #21 best restaurant in Barcelona by Tr..."
3,romantic dinner in paris,paris,None,None,5,hf_llm,True,None,True,[],True,1.0,"[Jean Yves of Chef Jean-Yves' Table, Restaurant de la Tour, We love italy, Pasta, Pizza & Piadine, Paris, Antoine, For The Love Of Food]","Given your preference for a romantic dinner in Paris, I recommend:\n\n1. **Jean Yves of Chef Jean-Yves' Table**: This highly-rated fine dining establishment offers an unforgettable culinary experience with its gourmet menu and attentive service. It's located in the heart of Paris, making it perfect for a romantic evening out.\n\n2. **Restaurant de la Tour**: Known for its exquisite cuisine and..."
4,family friendly seafood place in lisbon,lisbon,None,None,5,hf_llm,True,None,True,[],True,NaN,"[Solar 31 - Peixe e Marisco, A Taberna do Mar, Invicta, O Chefe e o Mar, Frade dos Mares]","Based on your request for a family-friendly seafood place in Lisbon, I recommend Solar 31 - Peixe e Marisco and A Taberna do Mar. Both are highly rated, offer delicious seafood, and are located in Lisbon. Additionally, you might consider Invicta, which is also highly recommended and features Mediterranean cuisine. All three options meet your criteria for being family-friendly and serving seafo..."
5,coffee and breakfast in athens,athens,None,None,5,hf_llm,True,None,True,[],True,1.0,"[Coffee Joint, Victory Cafe, Kekkos Traditional Cafe and Pastry, Aioli Coffee Snack Bar, 1937 Coffee Bar]","Based on your request for coffee and breakfast in Athens, I recommend:\n\n1. **Coffee Joint** - A highly-rated establishment that offers both coffee and breakfast options. It's known for its affordable prices and Mediterranean cuisine, making it a great choice for a quick caffeine fix and a hearty start to your day.\n\n2. **Victory Cafe** - Another top-rated option, Victory Cafe provides a var..."
6,expensive fine dining in milan,milan,None,expensive,5,hf_llm,True,None,True,[],True,NaN,"[Seta, Ba Restaurant, LUME, L'ALCHIMIA Ristorante & Lounge Bar, Langosteria Café]","Based on the 

In [ ]:
for q, out in hf_batch_outputs.items():
    print("=" * 120)
    print("QUERY:", q)
    print("PARSED:", json.dumps(out["parsed_query"], ensure_ascii=False))
    print("METHOD:", out["generation_method"])

    print("\nALLOWED RESTAURANTS:")
    for name in out["rag_context"]["allowed_restaurant_names"]:
        print("-", name)

    print("\nHF LLM ANSWER:")
    print(out["llm_answer"])

    print("\nFINAL ANSWER USED BY SYSTEM:")
    print(out["answer"])

    print("\nFINAL GROUNDED:", out["final_groundedness"]["is_grounded"])

QUERY: cheap italian restaurant in rome
PARSED: {"original_query": "cheap italian restaurant in rome", "normalized_query": "cheap italian restaurant in rome", "city": "rome", "country": null, "price_bucket": "cheap", "tags": ["italian"], "dietary": [], "matched_meals": [], "matched_features": []}
METHOD: hf_llm

ALLOWED RESTAURANTS:
- Pizza & Friends
- Stefano of Rome
- Ristorante Pizzeria Andrea
- Sto Bene Roma
- New Bar

HF LLM ANSWER:
Given your request for a cheap Italian restaurant in Rome, I recommend:

1. **Pizza & Friends** - This highly-rated establishment offers an authentic Italian experience without breaking the bank. Known for its delicious pizzas and friendly service, it's perfect for a quick lunch or dinner.

2. **Stefano of Rome** - Another top choice, this restaurant provides excellent value for money with its affordable prices and extensive menu options including vegetarian and vegan dishes. It’s known for its cozy atmosphere and attentive staff.

3. **Ristorante Pizz

In [ ]:
num_queries = len(hf_batch_df)

hf_llm_called_rate = float(hf_batch_df["llm_called"].mean()) if num_queries > 0 else None

hf_llm_grounded_rate = (
    float(hf_batch_df["llm_is_grounded"].dropna().mean())
    if hf_batch_df["llm_is_grounded"].notna().any()
    else None
)

hf_final_grounded_rate = (
    float(hf_batch_df["final_is_grounded"].mean())
    if num_queries > 0
    else None
)

hf_fallback_due_to_ungrounded = int(
    (hf_batch_df["generation_method"] == "template_fallback_due_to_ungrounded_hf_llm").sum()
)

hf_fallback_due_to_error = int(
    (hf_batch_df["generation_method"] == "template_fallback_due_to_hf_error").sum()
)

hf_template_no_results = int(
    (hf_batch_df["generation_method"] == "template_no_results").sum()
)

hf_success_count = int(
    (hf_batch_df["generation_method"] == "hf_llm").sum()
)

hf_rag_metrics = {
    "num_test_queries": num_queries,
    "hf_llm_called_rate": hf_llm_called_rate,
    "hf_llm_grounded_rate_before_fallback": hf_llm_grounded_rate,
    "final_grounded_answer_rate_after_fallback": hf_final_grounded_rate,
    "hf_llm_success_count": hf_success_count,
    "fallback_due_to_ungrounded_hf_llm": hf_fallback_due_to_ungrounded,
    "fallback_due_to_hf_error": hf_fallback_due_to_error,
    "template_no_results_count": hf_template_no_results,
}

hf_rag_metrics_df = pd.DataFrame([hf_rag_metrics])
display(hf_rag_metrics_df)

,num_test_queries,hf_llm_called_rate,hf_llm_grounded_rate_before_fallback,final_grounded_answer_rate_after_fallback,hf_llm_success_count,fallback_due_to_ungrounded_hf_llm,fallback_due_to_hf_error,template_no_results_count
0,11,0.818182,1.0,1.0,9,0,0,2


In [ ]:
def count_unsupported_mentions(x):
    if x is None:
        return 0

    if isinstance(x, list):
        return len(x)

    return 0


hf_batch_df["num_hf_unsupported_mentions"] = hf_batch_df["llm_unsupported_mentions"].apply(count_unsupported_mentions)

hf_hallucination_metrics = {
    "num_test_queries": len(hf_batch_df),
    "total_unsupported_restaurant_mentions_by_hf_llm": int(hf_batch_df["num_hf_unsupported_mentions"].sum()),
    "queries_with_hf_llm_hallucination": int((hf_batch_df["num_hf_unsupported_mentions"] > 0).sum()),
    "hf_llm_hallucination_query_rate": float((hf_batch_df["num_hf_unsupported_mentions"] > 0).mean()),
    "final_answers_with_unsupported_mentions": int((~hf_batch_df["final_is_grounded"]).sum()),
    "final_hallucination_rate_after_fallback": float((~hf_batch_df["final_is_grounded"]).mean()),
}

hf_hallucination_metrics_df = pd.DataFrame([hf_hallucination_metrics])
display(hf_hallucination_metrics_df)

,num_test_queries,total_unsupported_restaurant_mentions_by_hf_llm,queries_with_hf_llm_hallucination,hf_llm_hallucination_query_rate,final_answers_with_unsupported_mentions,final_hallucination_rate_after_fallback
0,11,0,0,0.0,0,0.0


In [ ]:
def compute_final_answer_constraint_coverage_hf(out):
    results = out["retrieval_output"]["results"]

    if len(results) == 0:
        return None

    parsed = out["parsed_query"]
    answer = out["answer"]

    constraints = []

    if parsed.get("city"):
        constraints.append(parsed["city"])

    if parsed.get("country"):
        constraints.append(parsed["country"])

    if parsed.get("price_bucket"):
        constraints.append(parsed["price_bucket"])

    constraints.extend(parsed.get("tags", []))
    constraints.extend(parsed.get("dietary", []))
    constraints.extend(parsed.get("matched_meals", []))
    constraints.extend(parsed.get("matched_features", []))

    if not constraints:
        return None

    answer_norm = normalize_text(answer)

    hits = []

    for constraint in constraints:
        constraint_norm = normalize_text(constraint)
        if constraint_norm:
            hits.append(contains_phrase(answer_norm, constraint_norm))

    if not hits:
        return None

    return float(np.mean(hits))


hf_coverage_rows = []

for q, out in hf_batch_outputs.items():
    hf_coverage_rows.append({
        "query": q,
        "generation_method": out["generation_method"],
        "num_results": len(out["retrieval_output"]["results"]),
        "constraint_coverage_in_final_answer": compute_final_answer_constraint_coverage_hf(out),
    })

hf_constraint_coverage_df = pd.DataFrame(hf_coverage_rows)
display(hf_constraint_coverage_df)

print(
    "Mean constraint coverage:",
    hf_constraint_coverage_df["constraint_coverage_in_final_answer"].dropna().mean()
)

,query,generation_method,num_results,constraint_coverage_in_final_answer
0,cheap italian restaurant in rome,hf_llm,5,1.000000
1,cheap vegetarian italian restaurant in rome,hf_llm,5,0.750000
2,vegetarian brunch in barcelona,hf_llm,5,0.666667
3,romantic dinner in paris,hf_llm,5,1.000000
4,family friendly seafood place in lisbon,hf_llm,5,0.666667
5,coffee and breakfast in athens,hf_llm,5,1.000000
6,expensive fine dining in milan,hf_llm,5,1.000000
7,vegan restaurant near berlin with outdoor seating,template_no_results,0,NaN
8,tapas place in madrid,hf_llm,5,1.000000
9,family restaurant in london,template_no_results,0,NaN


Mean constraint coverage: 0.8425925925925926


In [ ]:
HF_RAG_BATCH_SUMMARY_PATH = OUTPUT_DIR / "hf_rag_batch_summary.csv"
HF_RAG_METRICS_PATH = OUTPUT_DIR / "hf_rag_metrics.csv"
HF_HALLUCINATION_METRICS_PATH = OUTPUT_DIR / "hf_hallucination_metrics.csv"
HF_CONSTRAINT_COVERAGE_PATH = OUTPUT_DIR / "hf_constraint_coverage.csv"

hf_batch_df.to_csv(HF_RAG_BATCH_SUMMARY_PATH, index=False)
hf_rag_metrics_df.to_csv(HF_RAG_METRICS_PATH, index=False)
hf_hallucination_metrics_df.to_csv(HF_HALLUCINATION_METRICS_PATH, index=False)
hf_constraint_coverage_df.to_csv(HF_CONSTRAINT_COVERAGE_PATH, index=False)

print("Saved:")
print("-", HF_RAG_BATCH_SUMMARY_PATH)
print("-", HF_RAG_METRICS_PATH)
print("-", HF_HALLUCINATION_METRICS_PATH)
print("-", HF_CONSTRAINT_COVERAGE_PATH)

Saved:
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/hf_rag_batch_summary.csv
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/hf_rag_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/hf_hallucination_metrics.csv
- /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/hf_constraint_coverage.csv


In [ ]:
HF_RAG_ANSWERS_PATH = OUTPUT_DIR / "hf_rag_answers_examples.json"

hf_answers_payload = []

for q, out in hf_batch_outputs.items():
    hf_answers_payload.append({
        "query": q,
        "parsed_query": out["parsed_query"],
        "generation_method": out["generation_method"],
        "allowed_restaurant_names": out["rag_context"]["allowed_restaurant_names"],
        "hf_llm_answer": out["llm_answer"],
        "template_answer": out["template_answer"],
        "final_answer": out["answer"],
        "llm_error": out["llm_error"],
        "hf_llm_groundedness": out["llm_groundedness"],
        "final_groundedness": out["final_groundedness"],
        "num_results": len(out["retrieval_output"]["results"]),
    })

with open(HF_RAG_ANSWERS_PATH, "w", encoding="utf-8") as f:
    json.dump(hf_answers_payload, f, indent=2, ensure_ascii=False)

print("Saved:", HF_RAG_ANSWERS_PATH)

Saved: /content/drive/MyDrive/tablewise/artifacts_new/rag_generation/hf_rag_answers_examples.json


In [ ]:
def tablewise_answer_with_hf_llm(
    query,
    top_k=5,
    max_recommendations=3,
    verbose=False,
):

    return generate_safe_hf_rag_answer(
        query=query,
        top_k=top_k,
        max_recommendations=max_recommendations,
        verbose=verbose,
    )

In [ ]:
demo_queries_hf = [
    "cheap vegetarian italian restaurant in rome",
    "expensive fine dining in milan",
    "vegetarian brunch in barcelona",
    "tapas place in madrid",
    "family restaurant in london",
]

for q in demo_queries_hf:
    out = tablewise_answer_with_hf_llm(
        q,
        top_k=5,
        max_recommendations=3,
        verbose=False,
    )

    print("=" * 120)
    print("USER:", q)
    print("GENERATION METHOD:", out["generation_method"])
    print("TABLEWISE:")
    print(out["answer"])
    print("Grounded:", out["final_groundedness"]["is_grounded"])

    if out["llm_groundedness"] is not None:
        print("HF LLM unsupported mentions:", out["llm_groundedness"]["unsupported_mentions"])

USER: cheap vegetarian italian restaurant in rome
GENERATION METHOD: hf_llm
TABLEWISE:
Given your criteria for a cheap vegetarian Italian restaurant in Rome, I recommend Sfiziarte - The Art of Food and Pizza & Friends. Both offer excellent vegetarian options and are located in Rome, Italy. Sfiziarte's rating of 5.0/5 and its popularity as the #27 best restaurant in Rome make it a top choice. Pizza & Friends has a similar high rating (5.0/5) and offers both vegetarian and vegan options, making it another great option.
Grounded: True
HF LLM unsupported mentions: []
USER: expensive fine dining in milan
GENERATION METHOD: hf_llm
TABLEWISE:
Based on your request for an expensive fine dining experience in Milan, I recommend Seta, Ba Restaurant, and LUME. Seta offers exquisite Italian cuisine with a focus on seafood and Mediterranean flavors, while Ba Restaurant provides a unique blend of Chinese and Italian dishes. LUME combines traditional Italian cuisine with international influences, maki

In [ ]:
hf_test_query = "can you recommand me a cheap restaurant in barcelona?"

hf_test_output = generate_safe_hf_rag_answer(
    query=hf_test_query,
    top_k=5,
    max_recommendations=3,
    verbose=True,
)

print("Generation method:", hf_test_output["generation_method"])

if hf_test_output["llm_error"]:
    print("\nLLM error:")
    print(hf_test_output["llm_error"])

print("\nAllowed restaurant names:")
print(hf_test_output["rag_context"]["allowed_restaurant_names"])

print("\nHF LLM answer:")
print(hf_test_output["llm_answer"])

print("\nFinal answer used:")
print(hf_test_output["answer"])

print("\nHF groundedness:")
print(json.dumps(hf_test_output["llm_groundedness"], indent=2, ensure_ascii=False))

print("\nFinal groundedness:")
print(json.dumps(hf_test_output["final_groundedness"], indent=2, ensure_ascii=False))

Parsed query:
{
  "original_query": "can you recommand me a cheap restaurant in barcelona?",
  "normalized_query": "can you recommand me a cheap restaurant in barcelona",
  "city": "barcelona",
  "country": null,
  "price_bucket": "cheap",
  "tags": [],
  "dietary": [],
  "matched_meals": [],
  "matched_features": []
}

Initial rows: 1033798
City filter city=barcelona: 1033798 -> 10283
After location filter: 10283
Metadata score > 0 filter: 10283 -> 1582
After metadata filter/scoring: 1582
Computing semantic scores for 1,582 candidates...
Generation method: hf_llm

Allowed restaurant names:
['Conesa Entrepans', 'Yesterday Restaurant', 'My Bar', 'Pizza Circus', 'La Ronería']

HF LLM answer:
Based on your preferences, I recommend Conesa Entrepans, Yesterday Restaurant, and My Bar. Conesa Entrepans offers quick bites and bar options, while Yesterday Restaurant provides Mediterranean and Spanish cuisine with vegetarian-friendly dishes. My Bar is known for its affordable prices and serves a